# Walkthrough Klien SDK Terpadu (KF-13)

Notebook ini memenuhi KF-13: satu *environment* Python yang dikelola `uv` memuat dan menjalankan SDK Feast, SDK MLflow, dan klien *inference* KServe v2. Tiga blok di bawah adalah operasi baca yang independen, bukan satu rantai paksa. Feast mengembalikan nilai fitur *online*, MLflow membaca daftar *run*, dan klien KServe v2 memanggil *endpoint* `infer`.

Predictor `crypto-predictor` melayani model *bootstrap* iris (regresi logistik sklearn empat fitur, lihat `manifests/base/inferenceservices/inference.yaml`). Karena itu blok KServe adalah *smoke test* jalur penyajian v2, bukan prediksi kripto. Bridge promosi model menukar `storageUri` ke model nyata setelah *pipeline* pelatihan pertama selesai.

Jalankan dari `use-case-crypto/notebooks/`: `uv sync` lalu `uv run jupyter nbconvert --to notebook --execute unified_sdk_walkthrough.ipynb`. Prasyarat: akses cluster (repo Feast, MLflow tracking, endpoint KServe). Tanpa akses itu tiap blok mencetak diagnosa singkat, bukan keluaran palsu.

In [ ]:
import os

# In-cluster defaults mirror the services.
FEAST_REPO_PATH = os.getenv("FEAST_REPO_PATH", "/app/feature_store")
MLFLOW_TRACKING_URI = os.getenv(
    "MLFLOW_TRACKING_URI", "http://mlflow.model-lifecycle.svc.cluster.local:5000"
)
INFERENCE_URL = os.getenv(
    "INFERENCE_URL",
    "http://crypto-predictor-predictor.use-case-crypto.svc.cluster.local",
)
MODEL_NAME = os.getenv("MODEL_NAME", "crypto-predictor")
SYMBOL = os.getenv("SYMBOL", "BTC-USD")

print("FEAST_REPO_PATH    :", FEAST_REPO_PATH)
print("MLFLOW_TRACKING_URI:", MLFLOW_TRACKING_URI)
print("INFERENCE_URL      :", INFERENCE_URL)
print("MODEL_NAME         :", MODEL_NAME)

## 1. SDK Feast: get_online_features

Sama seperti `services/dashboard/ml-bridge/services/feature.py`: buka `FeatureStore(repo_path)`, kumpulkan referensi dari tiap *online feature view*, lalu baca nilai *online* untuk satu entity `symbol`.

In [ ]:
from feast import FeatureStore

try:
    store = FeatureStore(repo_path=FEAST_REPO_PATH)
    online_views = [v for v in store.list_feature_views() if getattr(v, "online", True)]
    feature_refs = [f"{v.name}:{f.name}" for v in online_views for f in v.features]
    if feature_refs:
        result = store.get_online_features(
            features=feature_refs,
            entity_rows=[{"symbol": SYMBOL}],
        )
        print({k: val[0] for k, val in result.to_dict().items()})
    else:
        print("Feast: no online feature views registered.")
except Exception as exc:
    print(f"Feast unreachable ({type(exc).__name__}): {exc}")

## 2. SDK MLflow: set_tracking_uri dan search_runs

Sama seperti `platform/services/trainer/src/deploy_kserve.py`: arahkan *tracking URI* lalu baca *run* berstatus `FINISHED`, terurut dari yang terbaru.

In [ ]:
import mlflow

try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    experiments = mlflow.search_experiments()
    runs = mlflow.search_runs(
        experiment_ids=[e.experiment_id for e in experiments],
        filter_string="status = 'FINISHED'",
        max_results=5,
        order_by=["start_time DESC"],
    )
    cols = [c for c in ("run_id", "experiment_id", "start_time") if c in runs.columns]
    print(runs[cols].to_string(index=False) if len(runs) else "MLflow: no FINISHED runs.")
except Exception as exc:
    print(f"MLflow unreachable ({type(exc).__name__}): {exc}")

## 3. Klien KServe v2: endpoint infer

Klien HTTP `httpx` (sama seperti `services/dashboard/ml-bridge/services/prediction.py`) memanggil *endpoint* Open Inference Protocol v2. *Payload* iris disalin persis dari probe `platform/components/model-serving/kserve/demo-health-check.yaml`. Nama model `crypto-predictor` berasal dari nama dasar `predictor` plus `namePrefix: crypto-`.

In [ ]:
import httpx

# Iris payload copied from health-check probe.
payload = {
    "inputs": [
        {
            "name": "input-0",
            "shape": [2, 4],
            "datatype": "FP32",
            "parameters": {"content_type": "np"},
            "data": [6.8, 2.8, 4.8, 1.4, 6.0, 3.4, 4.5, 1.6],
        }
    ]
}
url = f"{INFERENCE_URL}/v2/models/{MODEL_NAME}/infer"

try:
    with httpx.Client(timeout=30.0) as client:
        resp = client.post(url, json=payload)
        resp.raise_for_status()
        print(resp.json())
except Exception as exc:
    print(f"KServe v2 unreachable ({type(exc).__name__}): {exc}")

## Ringkasan

Satu *environment* `uv` (lihat `pyproject.toml` dan `uv.lock`) memuat dan menjalankan ketiga klien tanpa konfigurasi tambahan per layanan. Inilah antarmuka Python terpadu KF-13: Feast, MLflow, dan klien KServe v2 dijangkau dari satu *environment* yang sama.